In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introduzione alle primitive

<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Consigliamo di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Perché Qiskit ha introdotto le primitive?
Analogamente ai primi tempi dei computer classici, quando gli sviluppatori dovevano manipolare direttamente i registri della CPU, la prima interfaccia per le QPU restituiva semplicemente i dati grezzi dall'elettronica di controllo.
Questo non era un problema quando le QPU si trovavano nei laboratori e consentivano l'accesso diretto solo ai ricercatori.
Riconoscendo che la maggior parte degli sviluppatori non dovrebbe né potrebbe familiarizzarsi con la decodifica di tali dati grezzi in 0 e 1, Qiskit ha introdotto `backend.run`, una prima astrazione per accedere alle QPU nel cloud. Ciò ha permesso agli sviluppatori
di lavorare su un formato di dati familiare e di concentrarsi sul quadro generale.

Con la diffusione sempre più ampia dell'accesso alle QPU e lo sviluppo di nuovi algoritmi quantistici,
è emersa nuovamente la necessità di un'astrazione di livello superiore. In risposta, Qiskit ha introdotto
l'interfaccia delle primitive, ottimizzata per due compiti fondamentali nello sviluppo di algoritmi quantistici:
la stima del valore di aspettazione (`Estimator`) e il campionamento di circuiti (`Sampler`). L'obiettivo è ancora una volta
aiutare gli sviluppatori a concentrarsi maggiormente sull'innovazione e meno sulla conversione dei dati. L'interfaccia delle primitive sostituisce l'interfaccia `backend.run`, poiché `Sampler` fornisce lo stesso accesso diretto all'hardware offerto da `backend.run`.

## Che cos'è una primitiva?
I sistemi di elaborazione sono costruiti su più livelli di astrazione. Le astrazioni ti permettono di concentrarti su un
particolare livello di dettaglio rilevante per il compito in questione. Più ti avvicini all'hardware,
più basso è il livello di astrazione necessario (ad esempio, potrebbe essere necessario spostare o manipolare dati a livello di istruzione CPU). Più complesso è il compito che vuoi eseguire,
più alto sarà il livello delle astrazioni (ad esempio, potresti usare una libreria di programmazione per eseguire
calcoli algebrici).

In questo contesto, una *primitiva* è la più piccola istruzione di elaborazione, il blocco costruttivo più semplice da cui
si può creare qualcosa di utile per un dato livello di astrazione.

I recenti progressi nel calcolo quantistico hanno aumentato la necessità di lavorare a livelli di astrazione più elevati.
Man mano che il settore si sposta verso unità di elaborazione quantistica (QPU) più grandi e flussi di lavoro più complessi, il focus si sposta dall'interazione con i segnali dei singoli qubit alla considerazione dei dispositivi quantistici come sistemi che eseguono compiti necessari.

I due compiti più comuni per i computer quantistici sono il campionamento degli stati quantistici e il calcolo dei valori di aspettazione.
Questi compiti hanno motivato la progettazione delle primitive Qiskit: **Estimator** e **Sampler**.

- Estimator calcola i valori di aspettazione degli osservabili rispetto agli stati preparati dai circuiti quantistici.
- Sampler campiona il registro di output dall'esecuzione di circuiti quantistici.

In sintesi, il modello computazionale introdotto dalle primitive Qiskit avvicina la programmazione quantistica di un passo
a dove si trova oggi la programmazione classica, dove il focus è meno sui dettagli hardware e più sui risultati
che si vogliono ottenere.

## Definizione e implementazioni delle primitive
Esistono due tipi di primitive Qiskit: le classi base e le loro implementazioni. Le primitive Estimator e Sampler sono definite da classi base primitive open source che risiedono nel Qiskit SDK (nel modulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). I provider (come Qiskit Runtime) possono utilizzare queste classi base per derivare le proprie implementazioni di Sampler e Estimator. La maggior parte degli utenti interagirà con le implementazioni dei provider, non con le primitive base.

### Classi base
Le primitive `Base` sono classi astratte che definiscono un'interfaccia comune per l'implementazione delle primitive. Tutte le altre classi nel modulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) ereditano da queste classi base. Gli sviluppatori dovrebbero usarle se sono interessati a creare il proprio modello di esecuzione basato su primitive per un provider specifico. Queste classi potrebbero essere utili anche per chi vuole eseguire elaborazioni altamente personalizzate e trova che le implementazioni esistenti delle primitive siano troppo semplici per le proprie esigenze. Gli utenti generali non utilizzeranno direttamente le classi base.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) e [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - Sebbene le primitive V1 siano ancora utilizzabili, queste guide si concentrano sulle primitive V2 perché sono le più recenti e le più comunemente usate.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) e [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Le primitive di riferimento Qiskit seguono queste specifiche di interfaccia.

<span id="implementations"></span>
### Implementazioni
Tutte le primitive sono create a partire dalle classi base; di conseguenza, hanno la stessa struttura e modalità d'uso generale. Ad esempio, il formato dell'input per tutte le primitive Estimator è lo stesso. Tuttavia, esistono differenze nelle implementazioni che le rendono uniche.

Di seguito sono riportate le implementazioni delle classi base delle primitive:

- Le [primitive Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) e [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), forniscono un'implementazione più sofisticata (ad esempio, includendo la mitigazione degli errori) come servizio basato su cloud. Questa implementazione delle primitive base viene utilizzata per accedere all'hardware IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) e [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Implementazioni di riferimento delle primitive che utilizzano il simulatore integrato in Qiskit. Sono costruite con il modulo Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), producendo risultati basati su simulazioni ideali del vettore di stato. Si accede a esse tramite Qiskit. Consulta la guida [Simulazione esatta con le primitive Qiskit](/guides/simulate-with-qiskit-sdk-primitives) per i dettagli sull'utilizzo.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) e [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Puoi usare queste classi per "avvolgere" qualsiasi risorsa di calcolo quantistico in una primitiva. Questo ti permette di scrivere codice in stile primitiva per provider che non dispongono ancora di un'interfaccia basata su primitive. Queste classi possono essere usate esattamente come Sampler e Estimator normali, tranne per il fatto che devono essere inizializzate con un argomento `backend` aggiuntivo per selezionare su quale computer quantistico eseguire. Si accede a esse tramite Qiskit. Consulta la guida sulle [primitive backend](/guides/get-started-with-backend-primitives) per ulteriori informazioni.

## Opzioni
Puoi passare opzioni alle primitive per personalizzarle in base alle tue esigenze. Sebbene l'interfaccia del metodo `run()` delle primitive sia comune a tutte le implementazioni, le opzioni non lo sono. Consulta i riferimenti API per una specifica implementazione di primitiva per conoscere le opzioni supportate.

Ad esempio, consulta gli argomenti [Opzioni di Estimator](/guides/estimator-options) e [Opzioni di Sampler](/guides/sampler-options) per conoscere le opzioni delle primitive Qiskit Runtime, oppure consulta i [riferimenti API di Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) per le opzioni delle primitive Qiskit Aer.

## Vantaggi delle primitive Qiskit
Con le primitive, gli utenti Qiskit possono scrivere codice quantistico per una QPU specifica senza dover gestire esplicitamente ogni dettaglio. Inoltre, grazie al livello aggiuntivo di astrazione, potresti riuscire ad accedere più facilmente
alle funzionalità hardware avanzate di un determinato provider. Ad esempio, con le primitive Qiskit Runtime,
puoi sfruttare i più recenti progressi nella mitigazione e soppressione degli errori attivando opzioni come il [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) della primitiva, anziché costruire la tua implementazione di queste tecniche.

Per i provider hardware, implementare le primitive in modo nativo significa poter offrire ai propri utenti un modo più "pronto all'uso" per accedere alle funzionalità hardware, come le tecniche avanzate di post-elaborazione. È quindi più facile per i tuoi utenti beneficiare delle migliori capacità del tuo hardware.
## Passi successivi
> **Tip:** - Comprendi l'[input e output delle primitive](/guides/primitive-input-output).
> - Consulta [esempi dettagliati](/guides/simulate-with-qiskit-sdk-primitives).
> - Metti in pratica le primitive seguendo la [lezione sulle funzioni di costo](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
> - Consulta [Crea un provider](/guides/create-a-provider) per scoprire come implementare le tue primitive Sampler e Estimator.
> - Consulta i [riferimenti API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Leggi [Migra alle primitive V2](/guides/v2-primitives).
> - Scopri le [primitive Qiskit Runtime](/guides/qiskit-runtime-primitives), utilizzate per eseguire circuiti su QPU IBM.